from previous notebook `06-pre-disambiguation-optimization` we found that the best hyperparameters combinations are:
- scorer: `token_set_ratio`
- score_cutoff: `70`
- processor: `light_normalizer`

we now proceed to make the canonical entities disambiguation checked by an LLM and we give the task to the LLM to give a role to each canonical entity when it is related to a person.

In [ ]:
%load_ext rich

%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
import re
import time
import unicodedata
from collections import Counter
from operator import itemgetter
from pathlib import Path
from typing import Iterable, Tuple

import requests
from tqdm import tqdm

import numpy as np
import pandas as pd
import requests
from more_itertools import flatten, unique_everseen


from aymurai.llm_providers import OllamaLLMProvider
from aymurai.meta.entities import CanonicalEntities, CanonicalEntity
from aymurai.utils.json_data import get_pretty, save_json, load_json
import aymurai.evaluation.metrics as aymurai_metrics

from rapidfuzz.fuzz import token_set_ratio
from rapidfuzz import process

# #**1** First Step: Prepare the ***gold_json*** and ***ner_preds_json***

## /document-extract endpoint output

In [ ]:
API_URL = os.getenv("DOCUMENT_API_BASE_URL", "http://127.0.0.1:8000")
ENDPOINT = f"{API_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv(
        "DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/disambiguation-eval/files"
    )
)
GOLD_JSON_ROOT = Path(
    os.getenv(
        "GOLD_JSON_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}
JSON_EXTENSION = {".json"}
REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT}")
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

gold_jsons = discover_documents(GOLD_JSON_ROOT, JSON_EXTENSION)
print(f"Discovered {len(gold_jsons)} gold JSON files.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            ENDPOINT,
            files=files,
            timeout=REQUEST_TIMEOUT_S,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

## Inference

In [ ]:
from itertools import chain
from operator import itemgetter
from typing import Any

from more_itertools import unique_everseen


# Function to make inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample})
    response.raise_for_status()
    return response.json()


def parse_prediction_labels(predictions: list[dict[str, Any]]) -> list[dict[str, str]]:
    """
    Parse prediction labels to extract unique aymurai_label and aymurai_alt_text pairs.

    Args:
        predictions (list[dict[str, Any]]): A list of prediction dictionaries.

    Returns:
        list[dict[str, str]]: A list of dictionaries containing unique aymurai_label and aymurai_alt_text pairs.
    """
    attrs_stream = (
        label.get("attrs") or {}
        for label in chain.from_iterable(pred.get("labels", ()) for pred in predictions)
    )

    unique_pairs = unique_everseen(
        (
            attrs.get("aymurai_label"),
            attrs.get("aymurai_alt_text"),
        )
        for attrs in attrs_stream
        if attrs.get("aymurai_label") and attrs.get("aymurai_alt_text")
    )

    return sorted(
        ({"aymurai_label": label, "text": text} for label, text in unique_pairs),
        key=itemgetter("aymurai_label", "text"),
    )

## ***gold_json*** and ***ner_preds_json*** for input to clusterization developement

In [ ]:
def extract_and_save_entities(
    document_paths: list[Path], output_path: Path, file_ending: str, target_label: str
) -> None:
    """
    Processes a list of documents to extract specific entities and saves
    each result as an individual JSON file.
    It has been optimized to handle both document files and pre-existing JSON files.
    """

    with requests.Session() as session:
        if ".json" not in document_paths[0].suffix:
            for doc_path in tqdm(
                document_paths, desc=f"Extracting {target_label} entities"
            ):
                doc_path = Path(doc_path)

                # 1. API Extraction
                response = call_extraction_api(session, doc_path)
                document_data = response.get("detail", {}).get("document")

                if not document_data:
                    print(f"Warning: No document content found for {doc_path.name}")
                    continue

                # 2. Processing
                raw_predictions = [
                    get_predictions(paragraph) for paragraph in document_data
                ]
                parsed_labels = parse_prediction_labels(raw_predictions)

                # 3. Filtering by dynamic label
                filtered_entities = [
                    item
                    for item in parsed_labels
                    if item.get("aymurai_label") == target_label
                ]

                # 4. Saving individual file
                clean_base_name = re.sub(
                    r"\s+|_", "-", os.path.splitext(os.path.basename(doc_path))[0]
                )
                clean_base_name = re.sub(r"-{2,}", "-", clean_base_name).strip("-")
                clean_name = re.sub(r"-{2,}", "-", clean_base_name)
                file_name = f"{clean_name}{file_ending}"
                save_path = output_path / file_name

                save_json(file_path=save_path, json_data=filtered_entities)
        else:
            for doc_path in tqdm(
                document_paths, desc=f"Extracting {target_label} entities"
            ):
                doc_path = Path(doc_path)

                # 1. Load existing JSON data
                document_data = load_json(doc_path)

                if not document_data:
                    print(f"Warning: No document content found for {doc_path.name}")
                    continue

                # 2. Filtering by dynamic label
                filtered_entities = [
                    item
                    for item in document_data
                    if item.get("aymurai_label") == target_label
                ]

                # 3. Saving individual file
                clean_base_name = re.sub(
                    r"\s+|_", "-", os.path.splitext(os.path.basename(doc_path))[0]
                )
                clean_base_name = re.sub(r"-{2,}", "-", clean_base_name).strip("-")
                clean_name = re.sub(r"-{2,}", "-", clean_base_name)
                file_name = f"{clean_name}{file_ending}"
                save_path = output_path / file_name

                save_json(file_path=save_path, json_data=filtered_entities)

In [ ]:
def cluster_with_cdist(
    items: list[dict],
    threshold: int = 90,
    scorer: callable = token_set_ratio,
    processor: callable = None,
):
    """
    Cluster entities and prepare them for CanonicalEntity conversion.
    """
    if not items:
        return []

    # 1. Extract texts and apply normalization
    # We keep track of the original text, the processed text, and the label
    entities = [item.get("text", "") for item in items]
    labels = [item.get("aymurai_label", "UNKNOWN") for item in items]

    if processor:
        normed = [processor(e) for e in entities]
    else:
        normed = [str(e) for e in entities]

    # 2. Similarity Matrix
    sim = process.cdist(normed, normed, scorer=scorer, score_cutoff=threshold)
    sim = np.array(sim)

    # 3. Union-Find Logic
    parent = list(range(len(normed)))

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        root_i, root_j = find(i), find(j)
        if root_i != root_j:
            parent[root_j] = root_i

    n = len(normed)
    for i in range(n):
        for j in range(i + 1, n):
            if sim[i, j] >= threshold:
                union(i, j)

    # 4. Group into the format parse_item expects: (orig, norm, label)
    clusters_map = {}
    for idx in range(n):
        root = find(idx)
        if root not in clusters_map:
            clusters_map[root] = []
        # This tuple matches your 'parse_item' len == 3 condition
        clusters_map[root].append((entities[idx], normed[idx], labels[idx]))

    return list(clusters_map.values())


def parse_item(item: tuple[str, ...]) -> tuple[str, str, str]:
    """
    Parse an item into (label, orig, norm).

    Accepts:
      - (orig, norm, label)
      - (labelled_orig, labelled_norm) with prefix 'LABEL:'
    Args:
        item (tuple[str, ...]): input item

    Returns:
        tuple[str, str, str]: (label, orig, norm)
    """
    if len(item) == 3:
        orig, norm, label = item
        return label, orig, norm

    # len == 2: assume "LABEL:text"
    labelled_orig, labelled_norm = item
    label, orig = labelled_orig.split(":", 1)
    _, norm = labelled_norm.split(":", 1)
    return label, orig, norm


def pick_cluster_label(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Pick the most common label from parsed items.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen label
    """
    labels = [lbl for lbl, _, _ in parsed_items]
    # majority vote; fallback to first
    return Counter(labels).most_common(1)[0][0]


def pick_canonical_text(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Choose the longest original surface form; tweak as needed.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen canonical text
    """
    return max(parsed_items, key=lambda x: len(x[1]))[1]


def clusters_to_canonical_entities(
    clusters: list[list[tuple[str, str]]],
) -> list[CanonicalEntity]:
    """
    Convert clusters to CanonicalEntity objects.

    Args:
        clusters (list[list[tuple[str, str]]]): clusters of (original, normalized) entity tuples

    Returns:
        list[CanonicalEntity]: list of CanonicalEntity objects
    """
    canonical_entities = []

    for cluster in clusters:
        parsed = [parse_item(item) for item in cluster]  # [(label, orig, norm), ...]
        label = pick_cluster_label(parsed)
        canonical_text = pick_canonical_text(parsed)
        aliases = sorted({orig for _, orig, _ in parsed})
        ce = CanonicalEntity(
            aymurai_label=label,
            canonical_text=canonical_text,
            aliases=aliases,
            attributes={},
            relations=[],
        )
        canonical_entities.append(ce)

    return canonical_entities


def light_normalizer(s: str) -> str:
    return s.lower().strip() if s else ""

## Canonical Entity extraction

In [ ]:
# Available models

models = [
    "phi4:14b",
    "gpt-oss:20b",
    "llama3",
    "llama3.1:8b",
    "gemma3:270m",
]

In [ ]:
# Sanity check
provider = OllamaLLMProvider(model=models[0])
provider.generate("hola, ¿cómo estás?")

In [ ]:
system_prompt_1 = """
Eres un asistente experto en análisis legal y anonimización de documentos judiciales.
Tu función es auditar y enriquecer una lista de **Entidades Canónicas (Personas)** pre-agrupadas.

# Material de Trabajo
Para tu análisis, recibirás:
1. **Contexto de la Causa:** Una selección de párrafos del documento original donde se detectaron las menciones.
2. **Entidades Canónicas:** Una lista de grupos de aliases donde cada grupo representa a una única persona física, en teoría.

# Tus Tareas
1. **Validación Semántica:** Usa el contexto para confirmar que los aliases de un grupo realmente refieren a la misma persona física. Si detectas que un grupo contiene personas distintas (ej. homónimos o familiares), sepáralos.
2. **Consolidación:** Si una persona aparece en dos grupos diferentes (ej. "Juan Pérez" en uno y "Juan Alberto Pérez" en otro), únelos en una única entidad canónica.
3. **Asignación de Rol:** Identifica el rol procesal (Juez/a, Denunciante, Imputado/a, Víctima, Abogado/a, etc.) analizando cómo interactúa la persona en los párrafos provistos.

# Reglas de Salida
- Devuelve **únicamente** un objeto JSON que contenga el array de entidades procesadas.
- `canonical_text`: El nombre más completo y formal encontrado.
- `attributes`: Diccionario con el campo `"role"`. Si no se detecta rol, que el diccionario attributes quede vacío. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Defensor/a de Cámara"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"
  Debes asignar un rol tal cual se escribe en esa lista, no inventes nuevos roles.
- Mantén los `aliases` originales tal cual aparecen en el texto.

# Estructura de Respuesta (JSON)
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Alias Principal",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]

# Ejemplo de Input
Contexto: "...la declaración del Sr. Juan Carlos Ruiz ante este Juzgado. El imputado Ruiz negó los cargos. Por su parte, la Dra. Elena Sosa, fiscal de la causa, solicitó..."
Entidades Candidatas:
[{
  "entity_id": "optional-unique-id",
  "aymurai_label": "PER",
  "canonical_text": "Juan Carlos Ruiz",
  "aliases": [
    "Juan Carlos Ruiz",
    "Ruiz"
  ]
},
{
  "entity_id": "optional-unique-id",
  "aymurai_label": "PER",
  "canonical_text": "Elena Sosa",
  "aliases": [
    "Elena Sosa"
  ]
}]

# Ejemplo de Output
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Juan Carlos Ruiz",
    "aliases": [
      "Juan Carlos Ruiz",
      "Ruiz"
    ],
    "attributes": {
      "role": "Imputado"
    }
  },
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Elena Sosa",
    "aliases": [
      "Elena Sosa"
    ],
    "attributes": {
      "role": "Fiscal"
    }
  }
]
"""

In [ ]:
system_prompt_2 = """
Eres un asistente experto en análisis legal y anonimización de documentos judiciales.
Tu función es auditar y enriquecer una lista de **Entidades Canónicas (Personas)** pre-agrupadas.

# Material de Trabajo
Para tu análisis, recibirás:
1. **Contexto de la Causa:** Una selección de párrafos del documento original donde se detectaron las menciones.
2. **Entidades Canónicas:** Una lista de grupos de aliases donde cada grupo representa a una única persona física, en teoría.

# Tus Tareas
1. **Validación Semántica:** Usa el contexto para confirmar que los aliases de un grupo realmente refieren a la misma persona física. Si detectas que un grupo contiene personas distintas (ej. homónimos o familiares), sepáralos.
2. **Consolidación:** Si una persona aparece en dos grupos diferentes (ej. "Juan Pérez" en uno y "Juan Alberto Pérez" en otro), únelos en una única entidad canónica.
3. **Asignación de Rol:** Identifica el rol procesal (Juez/a, Denunciante, Imputado/a, Víctima, Abogado/a, etc.) analizando cómo interactúa la persona en los párrafos provistos.

# Reglas de Salida
- Devuelve **únicamente** un objeto JSON que contenga el array de entidades procesadas.
- `canonical_text`: El nombre más completo y formal encontrado.
- `attributes`: Diccionario con el campo `"role"`. Si no se detecta rol, que el diccionario attributes quede vacío. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Defensor/a de Cámara"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"
  Debes asignar un rol tal cual se escribe en esa lista, no inventes nuevos roles.
- Mantén los `aliases` originales tal cual aparecen en el texto.

# Estructura de Respuesta (JSON)
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Alias Principal",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
user_prompt_template_1 = """
A continuación se proporcionan fragmentos relevantes de un documento judicial y una lista de entidades (personas) pre-clusterizadas por similitud de texto.

# Contexto de la causa, párrafos con alguna mención a las personas:
{document_text}

# Entidades Pre-clusterizadas a validar:
{canonical_entities}

# Instrucciones:
1. **Validación de Aliases:** Analiza si los aliases dentro de cada entidad canónica pertenecen realmente a la misma persona según los párrafos de contexto. 
   - Si un alias pertenece a una persona distinta (ej. un familiar con nombre similar), sepáralo en una nueva entidad.
   - Si dos grupos de entidades canónicas refieren a la misma persona, fusiónalos.
2. **Identificación de Roles:** Extrae el rol procesal de cada persona basándote en el contexto (ej: Juez/a, Fiscal, Denunciante, Imputado/a, Víctima, Abogado/a, Perito/a, Testigo).
3. **Normalización:** Define el `canonical_text` como el nombre más completo y formal que aparezca en los aliases, si este trabajo ya está bien hecho no lo modifiques.
4. **Formato:** Devuelve la lista final de entidades en el formato JSON solicitado en el system prompt, asegurándote de no omitir a nadie que sea relevante.
"""

In [ ]:
system_prompt_3 = """
Eres un asistente experto en desambiguación de entidades judiciales.
Recibirás fragmentos de una causa y un JSON de entidades canónicas pre-agrupadas.

### Tu Tarea:
1. **Validar y Fusionar:** Verifica que cada entidad represente a una única persona real. Si dos grupos de entidades refieren a la misma persona, fusiónalos.
2. **Limpiar Aliases:** Elimina prefijos (Dr., Sra., Lic., etc.) o sufijos de los nombres (Asesor, Administrativo, etc.). Los aliases deben ser solo nombres propios.
3. **Filtrar:** Elimina cualquier entidad que no sea una persona física.
4. **Asignar Rol:** Clasifica cada entidad según el contexto usando **únicamente** esta lista estandarizada:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
   - **Nota:** No añadas información extra (ej. "Juez de Cámara" debe ser solo "Juez/a"). Si no se identifica rol, usa `null`.

### Formato de Salida (JSON):
Devuelve exclusivamente un array de objetos:
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Alias Principal",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
system_prompt_4 = """
Eres un asistente experto en desambiguación de entidades judiciales.
Recibirás fragmentos de una causa y un JSON de entidades canónicas pre-agrupadas.

### Tu Tarea:
1. **Validar y Fusionar:** Verifica que cada entidad represente a una única persona real. Si dos grupos de entidades refieren a la misma persona, fusiónalos.
2. **Limpiar Aliases:** Elimina prefijos (Dr., Dra., Dres., Sra., Sr., Lic., etc.) o sufijos de los nombres (Asesor, Administrativo, etc.). Los aliases deben ser solo nombres propios.
3. **Filtrar:** Elimina cualquier entidad que no sea una persona física.
4. **Asignar Rol:** Clasifica cada entidad según el contexto usando **únicamente** esta lista estandarizada:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
   
   **Instrucciones críticas de rol:**
   - Si una persona es mencionada como "Dr.", "Dra." o "Dres." y el contexto no indica que es Juez, Fiscal o Defensor oficial, asígnale el rol de **"Abogado/a"**.
   - No añadas información extra (ej. "Juez de Cámara" debe ser solo "Juez/a"). Si no se identifica un rol de la lista, usa `null`.

### Formato de Salida (JSON):
Devuelve exclusivamente un array de objetos:
[
  {
    "canonical_text": "Nombre Completo Normalizado",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
system_prompt_5 = """
Eres un extractor de roles legales. Tu única función es auditar una lista de entidades pre-agrupadas basándote en fragmentos de texto judicial.

### Instrucciones Estrictas:
1. **Filtrar:** Elimina cualquier entidad que no sea una persona física (ej: calles, instituciones, leyes, fechas).
2. **Asignar Rol:** Identifica el rol procesal usando EXCLUSIVAMENTE esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
3. **Regla "Dr/a":** Si la persona es mencionada como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito (como Juez o Fiscal), asígnale "Abogado/a".
4. **Limpiar:** En `canonical_text` y `aliases`, elimina otras palabras que no sean nombres propios.
5. **Fusionar:** Si dos entidades candidatas refieren a la misma persona, devuélvelas como una sola.
"""

In [ ]:
system_prompt_6 = """
Eres un extractor de roles legales. Tu única función es auditar una lista de entidades pre-agrupadas basándote en fragmentos de texto judicial.

### Instrucciones Estrictas:
1. **Filtrar:** Elimina cualquier entidad que no sea una persona física (ej: calles, instituciones, leyes, fechas).
2. **Asignar Rol:** Identifica el rol procesal usando EXCLUSIVAMENTE esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
3. **Regla "Dr/a":** Si la persona es mencionada como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito (como Juez o Fiscal), asígnale "Abogado/a".
4. **Fusionar:** Si dos entidades candidatas refieren a la misma persona, devuélvelas como una sola.
5. No modifiques los `canonical_text` ni los `aliases` originales.
"""

In [ ]:
system_prompt_7 = """
Eres un extractor de roles legales. Tu única función es auditar una lista de entidades pre-agrupadas basándote en fragmentos de texto judicial.

### Instrucciones Estrictas:
1. **Filtrar:** Elimina cualquier entidad que no sea una persona física (ej: calles, instituciones, leyes, fechas).
2. **Asignar Rol:** Identifica el rol procesal usando EXCLUSIVAMENTE esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
3. **Regla "Dr/a":** Si la persona es mencionada como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito (como Juez o Fiscal), asígnale "Abogado/a".
4. **Manejo de Siglas e Iniciales:** Ten en cuenta que las personas pueden ser mencionadas por sus iniciales (ej: "M.L." para "Martín López"). Si el contexto permite confirmar que unas iniciales refieren a una persona ya identificada, trátalas como un alias de esa misma entidad.
5. **Fusionar:** Si dos entidades candidatas refieren a la misma persona (ya sea por nombre completo, apellido solo o iniciales), devuélvelas como una sola entidad unificada.
6. **Integridad:** No modifiques los `canonical_text` ni los `aliases` originales; mantén la literalidad de lo extraído por el NER.

Devuelve exclusivamente un array de objetos:
[
  {
    "canonical_text": "Nombre Completo Normalizado",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
user_prompt_template_2 = """
A continuación se proporcionan fragmentos relevantes de un documento judicial y una lista de entidades (personas) pre-clusterizadas por similitud de texto.

# Contexto de la causa, párrafos con alguna mención a las personas:
{document_text}

# Entidades Pre-clusterizadas a validar:
{canonical_entities}
"""

In [ ]:
system_prompts = [
    system_prompt_1,
    system_prompt_2,
    system_prompt_3,
    system_prompt_4,
    system_prompt_5,
    system_prompt_6,
    system_prompt_7,
]

user_prompt_templates = [
    user_prompt_template_1,
    user_prompt_template_2,
]

In [ ]:
def llm_infer_canonical_entities(
    system_prompt: str,
    user_prompt_template: str,
    model: str,
    document_path: Path,
    pre_cluster_path: Path,
    gold_path: Path,
    context_window_length: int = 120,
    model_context: int = 9_500,
    target_label: str = "PER"
) -> dict:
    
    """
    Infers canonical entities using an LLM by providing context windows 
    around detected PER entities and pre-clusterization results.
    """

    # 1. Filename cleaning
    clean_base_name = document_path.stem.replace("_", "-").replace(" ", "-")
    target_filename = re.sub(r"-{2,}", "-", clean_base_name).strip("-")

    print(f"Processing document: {target_filename}")

    # 2. Extract document text via API
    session = requests.Session()
    api_response = call_extraction_api(session, Path(document_path))
    document_paragraphs = api_response.get("detail", {}).get("document")

    if not document_paragraphs:
        raise ValueError("Document text is empty or not found for {target_filename}.")

    # 3. Extract context windows for target label
    context_windows = set()

    for paragraph in tqdm(document_paragraphs, desc="Extracting context"):
        preds = get_predictions(paragraph)
        doc_text = preds['document']
        
        for label in preds['labels']:
            label_type = label['attrs'].get('aymurai_label')
            
            # Filter: only PER labels
            if label_type == target_label:
                start = label['start_char']
                end = label['end_char']
                
                # Define window boundaries
                window_start = max(0, start - context_window_length)
                window_end = min(len(doc_text), end + context_window_length)

                # Extract and clean the snippet
                snippet = " ".join(doc_text[window_start:window_end].split())
                context_windows.add(snippet)

    context_windows = list(context_windows)
    
    # 4. Prepare Canonical Entities from Pre-cluster
    canonical_entities_pre_cluster = load_json(json_file_path=pre_cluster_path)

    # Clean entities to save tokens
    canonical_entities_prompt = [
        {
            k: v
            for k, v in ce.items()
            if k not in ("entity_id", "relations", "attributes") or v
        }
        for ce in canonical_entities_pre_cluster
    ]

    # 5. Build Final User Prompt
    user_prompt = user_prompt_template.format(
        document_text="\n".join(context_windows).strip(),
        canonical_entities=get_pretty(canonical_entities_prompt),
    )

    # 6. LLM Inference
    provider = OllamaLLMProvider(model=model)
    response = provider.generate(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0, "num_ctx": model_context},
        format=CanonicalEntities.model_json_schema(),
    )

    # 7. Parse and Validate Output
    raw_llm_output  = [
            output for output in json.loads(response.text)["canonical_entities"]
        ]

    canonical_entities_llm = [
        CanonicalEntity.model_validate(canonical_entity)
        for canonical_entity in raw_llm_output
    ]

    # 8. Load Gold Standard for reference
    canonical_entities_gold = load_json(json_file_path=gold_path)

    print(f"\nSummary for {target_filename}:")
    print(f"- Pre-cluster entities: {len(canonical_entities_pre_cluster)}")
    print(f"- LLM generated entities: {len(canonical_entities_llm)}")
    print(f"- Gold standard entities: {len(canonical_entities_gold)}")

    # Return structured results
    return {
        "pre_cluster_json": canonical_entities_pre_cluster,
        "llm_output_json": canonical_entities_llm,
        "gold_standard_json": canonical_entities_gold,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt
    }

In [ ]:
def get_document_paths(document_check: Path, pre_clusterization_root: Path) -> dict:
    """
    Generates the pre-cluster and gold JSON paths for a given document.
    
    Args:
        document_check (Path): The original document Path object.
        pre_clusterization_root (Path): The root directory for pre-clusterization data.
        
    Returns:
        dict: A dictionary containing the 'pre_cluster_path' and 'gold_path'.
    """
    # 1. Prepare target filename
    clean_base_name = document_check.stem.replace("_", "-").replace(" ", "-")
    target_filename = re.sub(r"-{2,}", "-", clean_base_name).strip("-")

    # 2. Define subdirectories
    pre_cluster_dir = pre_clusterization_root / 'best-pre-clusterization-per'
    gold_dir = pre_clusterization_root / 'gold-jsons'

    # 3. Build final paths
    pre_cluster_json_path = pre_cluster_dir / f"{target_filename}-ner-preds-pre-cluster.json"
    gold_json_path = gold_dir / f"{target_filename}-canonical-entities-gold.json"

    return {
        "target_filename": target_filename,
        "pre_cluster_path": pre_cluster_json_path,
        "gold_path": gold_json_path
    }

In [ ]:
document_check = documents[0]
PRE_CLUSTERIZATION_ROOT = GOLD_JSON_ROOT.parent / "pre-clusterization"
paths = get_document_paths(document_check, PRE_CLUSTERIZATION_ROOT)

In [ ]:
context_window_lengths = list(range(80, 200, 10))

model_context_lengths = [7_500, 8_000, 8_500, 9_000, 9_500]

In [ ]:
llm_response_dict = llm_infer_canonical_entities(
    system_prompt=system_prompts['system_prompt_7'],
    user_prompt_template=user_prompt_templates['user_prompt_template_2'],
    model=models[0],
    document_path=document_check,
    pre_cluster_path=paths["pre_cluster_path"],
    gold_path=paths["gold_path"],
    context_window_length=120,
    model_context=9_500,
    target_label="PER"
)

In [ ]:
llm_response_dict['llm_output_json']

In [ ]:
llm_response_dict['pre_cluster_json']

In [ ]:
llm_response_dict['gold_standard_json']

In [ ]:
print(llm_response_dict['user_prompt'])

In [ ]:
print(llm_response_dict['system_prompt'])

In [ ]:
import shutil

def get_name(obj):
    """Helper to extract a clean string name from a function or object."""
    if obj is None:
        return "None"
    if hasattr(obj, "__name__"):
        return obj.__name__
    # Fallback for complex objects or partials: remove memory addresses
    clean_name = re.sub(r" at 0x[0-9a-fA-F]+", "", str(obj))
    return clean_name.strip("<>").replace("cyfunction ", "").replace("function ", "")


def run_llm_grid_search(
    models: list,
    system_prompts: dict,
    user_prompt_templates: dict,
    context_window_lengths: list,
    model_context_lengths: list,
    documents: list[Path],
    target_label: str,
    base_output_path: Path,
):
    """
    Runs a grid search over hyperparameters using nested loops.
    """

    # To keep track of all results and find the best
    all_combinations_results = []

    for model in models:
        for system_prompt_name, system_prompt in system_prompts.items():
            for user_prompt_template_name, user_prompt_template in user_prompt_templates.items():
                for context_window_length in context_window_lengths:
                    for model_context in model_context_lengths:
                        
                        print(
                            f"\nRunning grid search with model: {model}, "
                            f"context_window_length: {context_window_length}, "
                            f"model_context: {model_context}"
                        )

                        # 1. Create a descriptive folder name with the cleaned names
                        combo_name = (
                            f"model-{model}-system_prompt-{system_prompt_name}-"
                            f"user_prompt_template-{user_prompt_template_name}-context_window-{context_window_length}-"
                            f"model_context-{model_context}"
                        )

                        combo_dir = base_output_path / combo_name
                        combo_dir.mkdir(parents=True, exist_ok=True)

                        metrics_results = {}
                        metrics_file_path = (
                            combo_dir / f"evaluation_metrics_{target_label}.json"
                        )

                        # List to collect scores for this specific combination
                        current_combo_scores = []

                        for document in documents:
                            
                            # Get document paths
                            paths = get_document_paths(document, PRE_CLUSTERIZATION_ROOT)

                            # Run LLM inference
                            llm_response_dict = llm_infer_canonical_entities(
                                system_prompt=system_prompt,
                                user_prompt_template=user_prompt_template,
                                model=model,
                                document_path=document,
                                pre_cluster_path=paths['pre_cluster_path'],
                                gold_path=paths['gold_path'],
                                context_window_length=context_window_length,
                                model_context=model_context,
                                target_label=target_label
                            )
                            # Extract predictions
                            ce_llm = llm_response_dict['llm_output_json']
                            
                            # Validate and prepare for saving
                            ce_llm = [
                                CanonicalEntity.model_validate(entity)
                                for entity in ce_llm
                            ]

                            ce_llm = [
                                entity.model_dump() | {"entity_id": entity.entity_id.hex}
                                for entity in ce_llm
                            ]

                            # Save the result as the llm-pred.json
                            target_filename = paths['target_filename']
                            llm_filename = f"{target_filename}-llm-pred.json"
                            llm_output_path = combo_dir / llm_filename

                            save_json(
                                file_path=llm_output_path, json_data=ce_llm
                            )

                            # Extract gold standard
                            ce_gold = llm_response_dict['gold_standard_json']
                            
                            # Validate and prepare for saving
                            ce_gold = [
                                CanonicalEntity.model_validate(entity)
                                for entity in ce_gold
                            ]

                            ce_gold = [
                                entity.model_dump() | {"entity_id": entity.entity_id.hex}
                                for entity in ce_gold
                            ]

                            len_llm = len(ce_llm)
                            len_gold = len(ce_gold)

                            # Evaluate metrics
                            score, metrics = aymurai_metrics.evaluate_disambiguation(
                                gold_json=ce_gold,
                                pred_json=ce_llm,
                                target_label=target_label,
                            )
                            current_combo_scores.append(score)
                            doc_id = target_filename
                            metrics_results[doc_id] = {
                                "label": target_label,
                                "metric_value": score,
                                "detailed_metrics": metrics,
                                "len_llm_predictions": len_llm,
                                "len_gold_standard": len_gold
                            }

                        # --- Calculate Average for this combination ---
                        avg_score = (
                            np.mean(current_combo_scores) if current_combo_scores else 0.0
                        )

                        # Add average to the results file for this folder
                        final_output = {
                            "average_combination_score": avg_score,
                            "document_details": metrics_results,
                        }

                        with open(metrics_file_path, "w") as f:
                            json.dump(final_output, f, indent=4)

                        # Store combination info for final ranking
                        all_combinations_results.append(
                            {
                                "name": combo_name,
                                "score": avg_score,
                                "params": {
                                    "model": model,
                                    "system_prompt": system_prompt_name,
                                    "user_prompt_template": user_prompt_template_name,
                                    "context_window_length": context_window_length,
                                    "model_context": model_context
                                },
                            }
                        )

    # Save all combinations results in a summary file
    summary_file_path = (
        base_output_path / f"results_summary_{target_label}.json"
    )
    with open(summary_file_path, "w") as f:
        json.dump(all_combinations_results, f, indent=4)

    # --- Find the best combination ---
    best_combo = max(all_combinations_results, key=lambda x: x["score"])

    print(f"\nGrid Search for {target_label} completed.")

    # We now copy the best combination folder to a new location with a standardized name
    src = Path(base_output_path) / best_combo["name"]
    # Combine the parent destination path with the new folder name
    new_name = f"best-llm-pred-{target_label.lower()}"
    dest = Path(base_output_path.parent) / new_name
    
    try:
        # copytree creates the destination directory with the 'new_name'
        shutil.copytree(src, dest)
        print(f"Successfully copied '{src.name}' to '{dest}'")
    except FileExistsError:
        print(f"Error: A folder named '{new_name}' already exists in '{base_output_path.parent}'")
    except Exception as e:
        print(f"An error occurred: {e}")    

    return best_combo

In [ ]:
models = [
    "phi4:14b",
    "gpt-oss:20b",
    "llama3",
    "llama3.1:8b",
    "gemma3:270m",
]

system_prompts = {
    "system_prompt_1": system_prompt_1,
    "system_prompt_2": system_prompt_2,
    "system_prompt_3": system_prompt_3,
    "system_prompt_4": system_prompt_4,
    "system_prompt_5": system_prompt_5,
    "system_prompt_6": system_prompt_6,
    "system_prompt_7": system_prompt_7
}


user_prompt_templates = {
    "user_prompt_template_1": user_prompt_template_1,
    "user_prompt_template_2": user_prompt_template_2,
}

context_window_lengths = list(range(80, 200, 10))

model_context_lengths = [7_500, 8_000, 8_500, 9_000, 9_500]

In [ ]:
models = [
    "phi4:14b",
]

system_prompts = {
    "system_prompt_7": system_prompt_7
}


user_prompt_templates = {
    "user_prompt_template_2": user_prompt_template_2,
}

context_window_lengths = [120]

model_context_lengths = [9_500]

In [ ]:
top_result = run_llm_grid_search(
    models=models,
    system_prompts=system_prompts,
    user_prompt_templates=user_prompt_templates,
    context_window_lengths=context_window_lengths,
    model_context_lengths=model_context_lengths,
    documents=documents[0:3],
    target_label="PER",
    base_output_path=GOLD_JSON_ROOT.parent / "llm-grid-search-results",
)

In [ ]:
print(
    f"The best hyperparameter combination is '{top_result['name']}' "
    f"with an average score of {top_result['score']:.4f}."
)